# 01 — Data Understanding

## Purpose

Load the five Maven hospital tables, standardize column names, inspect schema, identify missing values and duplicates, and confirm the encounter classes available for downstream analysis.

No source data is modified in this notebook.

In [24]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [25]:
import pandas as pd
from config import RAW_DATA_DIR
from data_preparation import load_raw_tables, parse_dates

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

tables = parse_dates(load_raw_tables(RAW_DATA_DIR))
tables.keys()

dict_keys(['patients', 'encounters', 'procedures', 'payers', 'organizations'])

## Table dimensions

In [26]:
summary = pd.DataFrame(
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
    }
    for name, df in tables.items()
).sort_values("table")

summary

,table,rows,columns,duplicate_rows
1,encounters,27891,14,0
4,organizations,1,8,0
0,patients,974,20,0
3,payers,10,7,0
2,procedures,47701,9,0


## Schema inspection

In [27]:
for name, df in tables.items():
    print(f"\n{'=' * 80}\n{name.upper()} — shape={df.shape}")
    print(df.dtypes.to_string())
    display(df.head(3))


PATIENTS — shape=(974, 20)
id                       str
birthdate     datetime64[us]
deathdate     datetime64[us]
prefix                   str
first                    str
last                     str
suffix                   str
maiden                   str
marital                  str
race                     str
ethnicity                str
gender                   str
birthplace               str
address                  str
city                     str
state                    str
county                   str
zip                  float64
lat                  float64
lon                  float64


,id,birthdate,deathdate,prefix,first,last,suffix,maiden,marital,race,ethnicity,gender,birthplace,address,city,state,county,zip,lat,lon
0,5605b66b-e92d-c16c-1b83-b8bf7040d51f,1977-03-19,NaT,Mrs.,Nikita578,Erdman779,NaN,Leannon79,M,white,nonhispanic,F,Wakefield Massachusetts US,510 Little Station Unit 69,Quincy,Massachusetts,Norfolk County,2186.0,42.290937,-70.975503
1,6e5ae27c-8038-7988-e2c0-25a103f01bfa,1940-02-19,NaT,Mr.,Zane918,Hodkiewicz467,NaN,NaN,M,white,nonhispanic,M,Brookline Massachusetts US,747 Conn Throughway,Boston,Massachusetts,Suffolk County,2135.0,42.308831,-71.063162
2,8123d076-0886-9007-e956-d5864aa121a7,1958-06-04,NaT,Mr.,Quinn173,Marquardt819,NaN,NaN,M,white,nonhispanic,M,Gardner Massachusetts US,816 Okuneva Extension Apt 91,Quincy,Massachusetts,Norfolk County,2170.0,42.265177,-70.967085



ENCOUNTERS — shape=(27891, 14)
id                                     str
start                  datetime64[us, UTC]
stop                   datetime64[us, UTC]
patient                                str
organization                           str
payer                                  str
encounterclass                         str
code                                 int64
description                            str
base_encounter_cost                float64
total_claim_cost                   float64
payer_coverage                     float64
reasoncode                         float64
reasondescription                      str


,id,start,stop,patient,organization,payer,encounterclass,code,description,base_encounter_cost,total_claim_cost,payer_coverage,reasoncode,reasondescription
0,32c84703-2481-49cd-d571-3899d5820253,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,3de74169-7f67-9304-91d4-757e0f3a14d2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,185347001,Encounter for problem (procedure),85.55,1018.02,0.00,NaN,NaN
1,c98059da-320a-c0a6-fced-c8815f3e3f39,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,d9ec2e44-32e9-9148-179a-1653348cc4e2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,308335008,Patient encounter procedure,142.58,2619.36,0.00,NaN,NaN
2,4ad28a3a-2479-782b-f29c-d5b3f41a001e,2011-01-03 14:32:11+00:00,2011-01-03 14:47:11+00:00,73babadf-5b2b-fee7-189e-6f41ff213e01,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185349003,Encounter for check up (procedure),85.55,461.59,305.27,NaN,NaN



PROCEDURES — shape=(47701, 9)
start                datetime64[us, UTC]
stop                 datetime64[us, UTC]
patient                              str
encounter                            str
code                               int64
description                          str
base_cost                          int64
reasoncode                       float64
reasondescription                    str


,start,stop,patient,encounter,code,description,base_cost,reasoncode,reasondescription
0,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,3de74169-7f67-9304-91d4-757e0f3a14d2,32c84703-2481-49cd-d571-3899d5820253,265764009,Renal dialysis (procedure),903,NaN,NaN
1,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,d9ec2e44-32e9-9148-179a-1653348cc4e2,c98059da-320a-c0a6-fced-c8815f3e3f39,76601001,Intramuscular injection,2477,NaN,NaN
2,2011-01-04 14:49:55+00:00,2011-01-04 15:04:55+00:00,d856d6e6-4c98-e7a2-129b-44076c63d008,2cfd4ddd-ad13-fe1e-528b-15051cea2ec3,703423002,Combined chemotherapy and radiation therapy (p...,11620,363406005.0,Malignant tumor of colon



PAYERS — shape=(10, 7)
id                         str
name                       str
address                    str
city                       str
state_headquartered        str
zip                    float64
phone                      str


,id,name,address,city,state_headquartered,zip,phone
0,b3221cfc-24fb-339e-823d-bc4136cbc4ed,Dual Eligible,7500 Security Blvd,Baltimore,MD,21244.0,1-877-267-2323
1,7caa7254-5050-3b5e-9eae-bd5ea30e809c,Medicare,7500 Security Blvd,Baltimore,MD,21244.0,1-800-633-4227
2,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,Medicaid,7500 Security Blvd,Baltimore,MD,21244.0,1-877-267-2323



ORGANIZATIONS — shape=(1, 8)
id             str
name           str
address        str
city           str
state          str
zip          int64
lat        float64
lon        float64


,id,name,address,city,state,zip,lat,lon
0,d78e84ec-30aa-3bba-a33a-f29a3a454662,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,2114,42.362813,-71.069187


## Missing-value profile

In [28]:
missing_profiles = {}

for name, df in tables.items():
    profile = (
        df.isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .rename("missing_pct")
        .to_frame()
    )
    missing_profiles[name] = profile
    print(f"\n{name.upper()}")
    display(profile.head(15))


PATIENTS


,missing_pct
suffix,97.843943
deathdate,84.188912
maiden,60.369610
zip,14.579055
marital,0.102669
prefix,0.000000
id,0.000000
birthdate,0.000000
first,0.000000
race,0.000000



ENCOUNTERS


,missing_pct
reasondescription,70.062027
reasoncode,70.062027
stop,0.000000
patient,0.000000
id,0.000000
start,0.000000
payer,0.000000
organization,0.000000
encounterclass,0.000000
code,0.000000



PROCEDURES


,missing_pct
reasoncode,77.451206
reasondescription,77.451206
start,0.000000
stop,0.000000
patient,0.000000
code,0.000000
encounter,0.000000
base_cost,0.000000
description,0.000000



PAYERS


,missing_pct
address,10.0
state_headquartered,10.0
city,10.0
zip,10.0
phone,10.0
name,0.0
id,0.0



ORGANIZATIONS


,missing_pct
id,0.0
name,0.0
address,0.0
city,0.0
state,0.0
zip,0.0
lat,0.0
lon,0.0


## Encounter class distribution

In [29]:
encounters = tables["encounters"]

class_col = "encounterclass" if "encounterclass" in encounters.columns else "encounter_class"
encounters[class_col].astype(str).str.lower().value_counts(dropna=False)

encounterclass
ambulatory    12537
outpatient     6300
urgentcare     3666
emergency      2322
wellness       1931
inpatient      1135
Name: count, dtype: int64

## Date coverage

In [30]:
if {"start", "stop"}.issubset(encounters.columns):
    pd.DataFrame({
        "min_start": [encounters["start"].min()],
        "max_start": [encounters["start"].max()],
        "min_stop": [encounters["stop"].min()],
        "max_stop": [encounters["stop"].max()],
    })

## Data Quality Review and Findings

Review of the five source tables identified a generally clean and structurally consistent dataset.

### Source Table Sizes

| Table | Rows | Columns |
|---|---:|---:|
| Encounters | 27,891 | 14 |
| Patients | 974 | 20 |
| Procedures | 47,701 | 9 |
| Payers | 10 | 7 |
| Organizations | 1 | 8 |

The encounter data covers approximately **January 2011 through February 2022**.

### Key Findings

- No duplicate encounter identifiers were identified.
- No duplicate patient identifiers were identified.
- No duplicate payer identifiers were identified.
- No complete duplicate rows were identified in the five source tables.
- All encounter start and stop timestamps were successfully parsed.
- All procedure start and stop timestamps were successfully parsed.
- No encounters or procedures were identified with a stop timestamp earlier than the corresponding start timestamp.
- Encounter cost fields were already stored as numeric values and did not require text-to-numeric conversion.
- The source columns required for the planned analysis were present.

### Encounter Class Distribution

The encounter table contains six encounter classifications:

| Encounter Class | Encounters |
|---|---:|
| Ambulatory | 12,537 |
| Outpatient | 6,300 |
| Urgent care | 3,666 |
| Emergency | 2,322 |
| Wellness | 1,931 |
| Inpatient | 1,135 |

Because the objective of this project is to estimate **30-day inpatient readmission risk**, subsequent modeling is restricted to the **1,135 inpatient encounters**.

### Missing-Data Observations

Missing values are concentrated primarily in fields that are not required as primary model predictors.

In the encounter table, approximately **70% of `REASONCODE` and `REASONDESCRIPTION` values are missing**. These fields are therefore not used as primary predictors in the current model.

In the procedure table, approximately **77% of `REASONCODE` and `REASONDESCRIPTION` values are missing**. These fields are also excluded from the primary predictor set.

Within the patient table:

- `SUFFIX` is missing for approximately 98% of patients;
- `DEATHDATE` is missing for approximately 84%, which is expected for patients without a recorded death;
- `MAIDEN` is missing for approximately 60%;
- ZIP code is missing for approximately 15%;
- marital status has only minimal missingness.

These fields either have contextual reasons for missing values or are not necessary for the planned readmission model.

### Data Quality Conclusion

No major structural issue was identified that prevents the analysis from proceeding. The primary considerations for subsequent analysis are:

1. limiting the analytical population to inpatient encounters;
2. correctly constructing the 30-day readmission outcome;
3. excluding encounters without a complete 30-day observation period;
4. treating missing prior-utilization history appropriately; and
5. avoiding variables whose availability would not match the intended prediction point.

Overall, the source data is sufficiently complete and internally consistent for the planned analysis. The next stage applies these rules and performs data cleaning and exploratory analysis.
